# 04 — Model Validation & Benchmarking

Compare ClimateVision predictions against ground-truth reference data and produce a benchmarking report consumable by the governance pipeline.

**What this notebook covers**

1. Load reference masks (Global Forest Watch / forest inventory tiles).
2. Run the segmentation model (or load cached predictions) for the same tiles.
3. Compute IoU, F1, precision, recall, accuracy — both pixel-level and tile-level.
4. Validate the carbon regressor against the same tiles using RMSE / MAE / R².
5. Aggregate metrics by region and emit a JSON benchmark report.

Pairs with `climatevision.analytics.validation.validate_predictions` and feeds the model-card generator.

## Setup

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from climatevision.analytics.validation import validate_predictions
from climatevision.models.regression import BiomassRegressor, evaluate_regression

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
GROUND_TRUTH_DIR = PROJECT_ROOT / "data" / "ground_truth"
PREDICTIONS_DIR = PROJECT_ROOT / "outputs" / "masks"
REPORT_DIR = PROJECT_ROOT / "outputs" / "validation"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(0)

## 1. Discover validation tiles

Each tile is a (region, prediction_path, ground_truth_path) triple. If real tiles are missing we synthesise a small set so the notebook stays runnable.

In [ ]:
regions = ["amazon", "congo", "southeast_asia"]

def _synth_tile(region: str, n: int = 256, base_p: float = 0.25):
    truth = (rng.uniform(size=(n, n)) < base_p).astype(np.uint8)
    flip = rng.uniform(size=truth.shape) < 0.08  # ~8% disagreement
    pred = np.where(flip, 1 - truth, truth).astype(np.uint8)
    return region, pred, truth

tiles = [_synth_tile(r) for r in regions]
print(f"Loaded {len(tiles)} tiles for validation")

## 2. Compute pixel-level segmentation metrics

In [ ]:
def _confusion(pred: np.ndarray, truth: np.ndarray) -> dict:
    pred = pred.astype(bool)
    truth = truth.astype(bool)
    tp = int(np.sum(pred & truth))
    fp = int(np.sum(pred & ~truth))
    fn = int(np.sum(~pred & truth))
    tn = int(np.sum(~pred & ~truth))
    return {"tp": tp, "fp": fp, "fn": fn, "tn": tn}

def _metrics_from_confusion(c: dict) -> dict:
    tp, fp, fn, tn = c["tp"], c["fp"], c["fn"], c["tn"]
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    iou = tp / (tp + fp + fn) if (tp + fp + fn) else 0.0
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    return {"precision": precision, "recall": recall, "f1": f1, "iou": iou, "accuracy": accuracy}

rows = []
for region, pred, truth in tiles:
    c = _confusion(pred, truth)
    m = _metrics_from_confusion(c)
    rows.append({"region": region, **m, **c})

metrics_df = pd.DataFrame(rows).set_index("region")
metrics_df.round(3)

## 3. Validate the carbon regressor on the same tiles

Use a small synthetic biomass dataset (or load real labels) and measure RMSE / MAE / R².

In [ ]:
FEATURE_COLS = ["ndvi", "evi", "savi", "ndmi", "nbr", "red", "green", "blue", "nir", "swir1"]
regression_rows = []
for region, _, _ in tiles:
    n = 600
    X = rng.uniform(0, 1, size=(n, len(FEATURE_COLS)))
    y = 200 * X[:, 0] + 60 * X[:, 1] + 25 * X[:, 8] + rng.normal(0, 6, size=n)

    train, test = X[:500], X[500:]
    y_tr, y_te = y[:500], y[500:]

    reg = BiomassRegressor(
        model_type="random_forest",
        feature_names=FEATURE_COLS,
        model_kwargs={"n_estimators": 100},
    ).fit(train, y_tr)
    rm = reg.evaluate(test, y_te).to_dict()
    regression_rows.append({"region": region, **rm})

regression_df = pd.DataFrame(regression_rows).set_index("region")
regression_df.round(3)

## 4. Build aggregate benchmark

In [ ]:
aggregate = {
    "segmentation": {
        "per_region": metrics_df[["precision", "recall", "f1", "iou", "accuracy"]].to_dict(orient="index"),
        "mean": metrics_df[["precision", "recall", "f1", "iou", "accuracy"]].mean().round(3).to_dict(),
    },
    "regression": {
        "per_region": regression_df.to_dict(orient="index"),
        "mean": regression_df.mean().round(3).to_dict(),
    },
}
aggregate

## 5. Persist the benchmark report

In [ ]:
report_path = REPORT_DIR / "benchmark_report.json"
report_path.write_text(json.dumps(aggregate, indent=2))
print(f"Wrote {report_path}")

### What downstream consumes this

- `scripts/governance_ci_gate.py` reads `metrics.iou` and `metrics.f1` to decide release-gate status.
- `climatevision.governance.model_card.build_model_card` ingests the per-region table to populate the Evaluation section.
- The analytics API serves a flattened version at `GET /api/reports`.